## Load data

In [2]:
# Load data
import pandas as pd

df_fasttext_aligned = pd.read_pickle('../data/words10_embeddings_fasttext_aligned.pkl')
df_fasttext = pd.read_pickle('../data/words10_embeddings_fasttext.pkl')
df_para = pd.read_pickle('../data/words10_embeddings_paraphrase-multilingual-MiniLM-L12-v2.pkl')
df_labse = pd.read_pickle('../data/words10_embeddings_LaBSE.pkl')

## Calculate average cosine similairity per embedding

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def calculate_avg_cosine(row):
    word_cols = row.index[2:]
    embeddings = [row[col] for col in word_cols if isinstance(row[col], np.ndarray)]
    
    if len(embeddings) < 2:
        return np.nan
    
    similarities = cosine_similarity(embeddings)
    upper_triangle = similarities[np.triu_indices_from(similarities, k=1)]
    
    return np.mean(upper_triangle)

def process_model_long(df_embeddings, model_name):
    sim = df_embeddings.apply(calculate_avg_cosine, axis=1)
    
    return pd.DataFrame({
        "language": df_embeddings["language"],
        "model": df_embeddings["model"],
        "embedding_model": model_name,
        "avg_cosine": sim
    })

final_df = pd.concat([
    process_model_long(df_fasttext_aligned, "fastText (aligned)"),
    process_model_long(df_fasttext, "fastText"),
    process_model_long(df_labse, "LaBSE"),
    process_model_long(df_para, "paraphrase-multilingual-MiniLM-L12-v2"),
])

In [6]:
final_df.to_csv('../data/cosine_similairity.csv')